# Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# Exploratory Data Analysis

In [2]:
df = pd.read_csv("/kaggle/input/bitcoinheist-ransomware-dataset/BitcoinHeistData.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/bitcoinheist-ransomware-dataset/BitcoinHeistData.csv'

In [ ]:
df.shape

: 

In [ ]:
df.head()

: 

In [ ]:
df.describe()

: 

In [ ]:
df["label"].value_counts()

: 

In [ ]:
categorical_cols = df.select_dtypes(include="object").columns

: 

In [ ]:
numerical_cols = df.select_dtypes(include="number").columns

: 

In [ ]:
df.isnull().sum()

: 

In [ ]:
df["address"].nunique()

: 

In [ ]:
old_df = df.copy()

: 

# Feature Engineering

In [ ]:
df.drop(columns=["address", "year", "day"], axis=1, inplace=True)

: 

In [ ]:
for col in df.columns[:-1]:
    df[col] = df[col]

: 

In [ ]:
X = df.drop(columns=["label"])
y = df["label"]

: 

In [ ]:
X.columns

: 

In [ ]:
df.head()

: 

## Number of Instances

In [ ]:
new_df = pd.DataFrame()
grouped = df.groupby("label")

: 

In [ ]:
new_df["num_of_instances"] = grouped.size()

: 

## Average

In [ ]:
df.columns

: 

In [ ]:
for col in X.columns:
    new_df[f"{col}_avg"] = grouped[col].mean()

: 

In [ ]:
new_df.shape

: 

In [ ]:
new_df.head()

: 

## Standard Deviation

In [ ]:
for col in X.columns:
    new_df[f"{col}_std"] = grouped[col].agg(np.std).fillna(0)

: 

## Minimum

In [ ]:
for col in X.columns:
    new_df[f"{col}_min"] = grouped[col].min()

: 

## Maximum

In [ ]:
for col in X.columns:
    new_df[f"{col}_max"] = grouped[col].max()

: 

In [ ]:
new_df = new_df.reset_index()
new_df.head()

: 

# Data Preprocessing

## Undersampling

In [ ]:
black_rows = df[df["label"] != "white"]

: 

In [ ]:
white_sample = df[df["label"] == "white"].sample(n=158587, random_state=42)

: 

In [ ]:
sampled_df = pd.concat([black_rows, white_sample]).sample(frac=1, random_state=50)
sampled_df.head()

: 

## Label Encoding

We do label encoding of:
> * White label: 0
> * Ransomware: 1

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Create a label encoder object
label_encoder = LabelEncoder()

# Encode the labels
sampled_df["label"] = label_encoder.fit_transform(sampled_df["label"])

# Map 'white' to 0 and other values to 1
sampled_df["label"] = sampled_df["label"].apply(
    lambda x: 0 if x == label_encoder.transform(["white"])[0] else 1
)

sampled_df["label"]

: 

## Z-score Anomaly Filteration

In [ ]:
z_score_df = pd.DataFrame()
for col in sampled_df.columns:
    z_score_df[f"{col}_z_score"] = (
        sampled_df[col] - sampled_df[col].mean()
    ) / sampled_df[col].std()
z_score_df.head()

: 

In [ ]:
import pandas as pd

# Filter the columns based on the condition
filtered_df = z_score_df[
    z_score_df.apply(lambda row: all(-3 <= val <= 3 for val in row), axis=1)
]

# Display the filtered columns' data
filtered_df.head()

: 

## Train Test Split

> Training data: 66%
> Test data: 33%

In [ ]:
X = sampled_df.drop("label", axis=1)
y = (sampled_df["label"] > 0).astype("int")

: 

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.33, shuffle=True
)

: 

## Standardization

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

: 

# Training

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.metrics import classification_report

: 

In [ ]:
model = Sequential(
    [
        Dense(232, activation="relu", input_shape=(X_train_scaled.shape[1],)),
        Dropout(0.2),
        Dense(128, activation="relu"),
        Dropout(0.2),
        Dense(64, activation="relu"),
        Dropout(0.2),
        Dense(1, activation="sigmoid"),
    ]
)

: 

In [ ]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

: 

In [ ]:
history = model.fit(
    X_train_scaled, y_train, epochs=50, batch_size=32, validation_split=0.2
)

: 

# Evaluation

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test)

: 

In [ ]:
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

: 

In [ ]:
plt.plot(history.history["accuracy"], label="accuracy")
plt.plot(history.history["val_accuracy"], label="val_accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(loc="lower right")
plt.show()

: 

In [ ]:
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype("int32")
print(classification_report(y_test, y_pred))

: 